# Solutions · 0. Atom, mass and charge balances as linear constraints

Worked solutions to the three exercises of [Lecture 0](../../notebooks/part1_conservation/00_elemental_mass_and_charge_conservation_matrices.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Atomic Formula Matrix & Mass-Balance Linear Dependence")
# Species order: [CH3OH, CO, H2O, CO2, H2] for methanol synthesis (Feed: 1 mol CO2 + 3 mol H2)
species = ["CH3OH", "CO", "H2O", "CO2", "H2"]
C_atom = np.array([
    [1.0, 1.0, 0.0, 1.0, 0.0],  # Carbon (C) atoms per molecule
    [4.0, 0.0, 2.0, 0.0, 2.0],  # Hydrogen (H) atoms per molecule
    [1.0, 1.0, 1.0, 2.0, 0.0],  # Oxygen (O) atoms per molecule
])
atomic_masses = np.array([12.011, 1.008, 15.999])  # [C, H, O] in g/mol
molar_masses = atomic_masses @ C_atom
w_feed = np.array([0.0, 0.0, 0.0, 1.0, 3.0])
d_atom = C_atom @ w_feed
feed_mass = float(molar_masses @ w_feed)

print(f"{'Species':>8s} | {'C':>3s} {'H':>3s} {'O':>3s} | {'Molar Mass [g/mol]':>18s} | {'Feed [mol]':>10s}")
print("-" * 54)
for sp, col, mw, wf in zip(species, C_atom.T, molar_masses, w_feed):
    print(f"{sp:>8s} | {int(col[0]):3d} {int(col[1]):3d} {int(col[2]):3d} | {mw:18.3f} | {wf:10.3f}")

C_augmented_mass = np.vstack([C_atom, molar_masses])
assert np.linalg.matrix_rank(C_atom) == 3
assert np.linalg.matrix_rank(C_augmented_mass) == 3, "Mass row m^T = a^T C must not increase rank!"
print(f"\nElemental feed d = {d_atom} mol | Total feed mass = {feed_mass:.3f} g | rank(C) = {np.linalg.matrix_rank(C_atom)}")
heading("Demonstration 2: Stoichiometric Reaction Matrix & SVD Nullspace Projection")
nu = np.array([
    [1.0, 0.0, 1.0, -1.0, -3.0],  # Rxn 1: CO2 + 3 H2 <-> CH3OH + H2O
    [0.0, 1.0, 1.0, -1.0, -1.0],  # Rxn 2: CO2 + H2   <-> CO    + H2O
])
U_c, s_c, Vt_c = np.linalg.svd(C_atom, full_matrices=True)
N_svd = Vt_c[3:, :].T  # Orthonormal basis for the 2-D nullspace N(C)
P_null = N_svd @ N_svd.T

# Project physical reaction vectors nu onto the SVD nullspace basis:
nu_projected = (P_null @ nu.T).T
print("Singular values of C_atom:", np.round(s_c, 6))
print("Stoichiometric nullspace residual ||C_atom @ nu^T||_max:", float(np.max(np.abs(C_atom @ nu.T))))
print("Subspace projection discrepancy ||nu - P_null nu||_max  :", float(np.max(np.abs(nu - nu_projected))))
assert np.allclose(C_atom @ nu.T, 0.0, atol=1e-15)
assert np.allclose(nu, nu_projected, atol=1e-14)
heading("Demonstration 3: Stoichiometric Reactor Benchmark vs. Raw Unconstrained GC Sensor")
xi_grid = np.array([[0.24, 0.08], [0.35, 0.12], [0.10, 0.20]])
for idx, xi in enumerate(xi_grid, start=1):
    w_state = w_feed + nu.T @ xi
    atom_res = float(np.max(np.abs(C_atom @ w_state - d_atom)))
    mass_now = float(molar_masses @ w_state)
    print(f"Operating Point {idx} (xi={xi}) | Effluent w={np.round(w_state, 3)} | Mass={mass_now:.3f} g | Atom err={atom_res:.1e}")
    assert np.allclose(C_atom @ w_state, d_atom, atol=1e-14)
    assert np.isclose(mass_now, feed_mass, rtol=1e-14)

rng = np.random.default_rng(42)
w_true = w_feed + nu.T @ np.array([0.24, 0.08])
w_noisy_gc = w_true + rng.normal(0.0, 0.05, size=5)
atom_violation = C_atom @ w_noisy_gc - d_atom
mass_violation = float(molar_masses @ w_noisy_gc) - feed_mass
print(f"\nNoisy raw GC elemental imbalance [dC, dH, dO]: {np.round(atom_violation, 4)} mol")
print(f"Noisy raw GC phantom mass created            : {mass_violation:+.4f} g (out of {feed_mass:.3f} g)")
assert abs(mass_violation) > 0.1
heading("Demonstration 4: Analyzer Noise Sweep & Affine Conservation Projection")
C_pinv = C_atom.T @ np.linalg.inv(C_atom @ C_atom.T)
unit_noise = rng.normal(0.0, 1.0, size=5)
print(f"{'Noise sigma [mol]':>18s} | {'Raw Atom Err [mol]':>18s} | {'Raw Mass Err [g]':>16s} | {'Projected Atom Err':>18s}")
print("-" * 78)
for sig in [0.001, 0.01, 0.05, 0.10]:
    w_raw = w_true + sig * unit_noise
    w_proj = w_raw - C_pinv @ (C_atom @ w_raw - d_atom)
    raw_a = float(np.max(np.abs(C_atom @ w_raw - d_atom)))
    raw_m = abs(float(molar_masses @ w_raw) - feed_mass)
    prj_a = float(np.max(np.abs(C_atom @ w_proj - d_atom)))
    print(f"{sig:18.3f} | {raw_a:18.5f} | {raw_m:16.5f} | {prj_a:18.2e}")
    assert prj_a < 1e-14
heading("Inside the Algorithm: By-Hand Affine Projection vs. the Library")
w_by_hand = w_noisy_gc - C_atom.T @ np.linalg.solve(C_atom @ C_atom.T, C_atom @ w_noisy_gc - d_atom)
w_library = linear_physics.equality_constrained_lstsq(np.eye(5), w_noisy_gc, C_atom, d_atom, lam=0.0)["w"]
diff_norm = float(np.max(np.abs(w_by_hand - w_library)))
print("by hand (pure NumPy):", np.round(w_by_hand, 6))
print("library (engml_2_applied)     :", np.round(w_library, 6))
print(f"Max digit-for-digit discrepancy: {diff_norm:.2e}")
assert diff_norm < 1e-13
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Construct the $4 \times 6$ elemental + charge conservation matrix $\mathbf{C}_{\mathrm{aq}}$ (rows: $\mathrm{C, H, O}$, and ionic charge $z$) for the 6-species aqueous carbonate system $[\mathrm{CO_2(aq)}, \mathrm{H_2O}, \mathrm{H^+}, \mathrm{OH^-}, \mathrm{HCO_3^-}, \mathrm{CO_3^{2-}}]$. Verify that all three equilibrium reactions ($\mathrm{CO_2 + H_2O \rightleftharpoons H^+ + HCO_3^-}$, $\mathrm{HCO_3^- \rightleftharpoons H^+ + CO_3^{2-}}$, and $\mathrm{H_2O \rightleftharpoons H^+ + OH^-}$) lie identically in $\mathcal{N}(\mathbf{C}_{\mathrm{aq}})$ and prove that $\mathrm{rank}(\mathbf{C}_{\mathrm{aq}}) = 3$ due to the oxidation-state identity $z = n_{\mathrm{H}} - 2n_{\mathrm{C}} - n_{\mathrm{O}}$.

In [2]:
heading("Exercise 1: Aqueous Carbonate Elemental & Electroneutrality Matrix")
# Species: [CO2, H2O, H+, OH-, HCO3-, CO3^2-]
C_aq = np.array([
    [1.0, 0.0, 0.0, 0.0, 1.0, 1.0],    # C
    [0.0, 2.0, 1.0, 1.0, 1.0, 0.0],    # H
    [2.0, 1.0, 0.0, 1.0, 3.0, 3.0],    # O
    [0.0, 0.0, 1.0, -1.0, -1.0, -2.0], # Ionic charge z = H - 2*C - O
])
nu_aq = np.array([
    [-1.0, -1.0, 1.0, 0.0,  1.0, 0.0], # CO2 + H2O <-> H+ + HCO3-
    [ 0.0,  0.0, 1.0, 0.0, -1.0, 1.0], # HCO3- <-> H+ + CO3^2-
    [ 0.0, -1.0, 1.0, 1.0,  0.0, 0.0], # H2O <-> H+ + OH-
])
rank_C = int(np.linalg.matrix_rank(C_aq))
null_res = float(np.max(np.abs(C_aq @ nu_aq.T)))
ox_check = float(np.max(np.abs(C_aq[3] - (C_aq[1] + 4.0 * C_aq[0] - 2.0 * C_aq[2]))))
print(f"Aqueous carbonate conservation matrix rank = {rank_C} (6 species - 3 invariants = 3 reactions)")
print(f"Stoichiometric nullspace residual ||C_aq @ nu_aq^T||_max = {null_res:.2e}")
print(f"Oxidation-state charge identity residual |z - (H + 4C - 2O)| = {ox_check:.2e}")
assert rank_C == 3 and null_res < 1e-15 and ox_check < 1e-15


Exercise 1: Aqueous Carbonate Elemental & Electroneutrality Matrix
Aqueous carbonate conservation matrix rank = 3 (6 species - 3 invariants = 3 reactions)
Stoichiometric nullspace residual ||C_aq @ nu_aq^T||_max = 0.00e+00
Oxidation-state charge identity residual |z - (H + 4C - 2O)| = 0.00e+00


**Discussion.** Because carbon, hydrogen, and oxygen preserve fixed formal oxidation states ($+4, +1, -2$) across all six carbonate species, the charge of every species equals $n_{\mathrm{H}} + 4n_{\mathrm{C}} - 2n_{\mathrm{O}}$, so the charge row is a combination of the three element rows, giving $\mathrm{rank}(\mathbf{C}_{\mathrm{aq}}) = 3$ and $R = 6 - 3 = 3$ independent reactions.

---

## Exercise 2

**Problem.** Suppose an uncalibrated pH/ion-selective electrode array reports the perturbed aqueous concentrations $\mathbf{w}_{\mathrm{bad}} = [0.010, 55.50, 10^{-4}, 10^{-10}, 0.0020, 0.0005]^\top\;\mathrm{mol/L}$. Compute the unphysical net ionic charge imbalance $\Delta z = \mathbf{c}_z^\top \mathbf{w}_{\mathrm{bad}}$ ($\mathrm{mol\;charge/L}$) and reconcile the vector onto the exact electroneutrality and elemental conservation manifold.

In [3]:
heading("Exercise 2: Electroneutrality Violation Audit & Conservation Projection")
w_aq_feed = np.array([0.012, 55.50, 0.0, 0.0, 0.0, 0.0])
d_aq = C_aq[:3] @ w_aq_feed
w_bad = np.array([0.010, 55.50, 1e-4, 1e-10, 0.0020, 0.0005])
charge_imbalance_before = float(C_aq[3] @ w_bad)
w_aq_rec = linear_physics.equality_constrained_lstsq(np.eye(6), w_bad, C_aq[:3], d_aq, lam=0.0)["w"]
charge_imbalance_after = float(C_aq[3] @ w_aq_rec)
print(f"Net ionic charge before reconciliation: {charge_imbalance_before:+.6f} mol/L (Violates electroneutrality)")
print(f"Net ionic charge after reconciliation : {charge_imbalance_after:+.2e} mol/L")
assert abs(charge_imbalance_before) > 1e-3 and abs(charge_imbalance_after) < 1e-14


Exercise 2: Electroneutrality Violation Audit & Conservation Projection
Net ionic charge before reconciliation: -0.002900 mol/L (Violates electroneutrality)
Net ionic charge after reconciliation : +4.88e-19 mol/L


**Discussion.** Enforcing the three independent elemental balances $\mathrm{C, H, O}$ automatically enforces exact aqueous electroneutrality $\mathbf{c}_z^\top \hat{\mathbf{w}} = 0$ to machine precision.

---

## Exercise 3

**Problem.** Implement a reusable function `stoichiometric_nullspace_basis(C: np.ndarray, tol: float = 1e-10) -> np.ndarray` using `np.linalg.svd` that returns an orthonormal basis matrix $\mathbf{N} \in \mathbb{R}^{S \times R}$ for $\mathcal{N}(\mathbf{C})$, and verify on both `C_atom` ($R=2$) and `C_aq` ($R=3$) that $\mathbf{C}\mathbf{N} = \mathbf{0}$ and $\mathbf{N}^\top\mathbf{N} = \mathbf{I}_R$.

In [4]:
heading("Exercise 3: Implement stoichiometric_nullspace_basis From Scratch")
def stoichiometric_nullspace_basis(C: np.ndarray, tol: float = 1e-10) -> np.ndarray:
    """Return an orthonormal basis N (S x R) for the right nullspace of conservation matrix C."""
    C_mat = np.atleast_2d(np.asarray(C, dtype=float))
    _, s_vals, Vt = np.linalg.svd(C_mat, full_matrices=True)
    rank = int(np.sum(s_vals > tol))
    return Vt[rank:, :].T

N_gas = stoichiometric_nullspace_basis(C_atom)
N_aqu = stoichiometric_nullspace_basis(C_aq)
print(f"Methanol gas system nullspace shape: {N_gas.shape} | ||C N||_max = {np.max(np.abs(C_atom @ N_gas)):.2e}")
print(f"Aqueous carbonate nullspace shape  : {N_aqu.shape} | ||C N||_max = {np.max(np.abs(C_aq @ N_aqu)):.2e}")
assert N_gas.shape == (5, 2) and np.allclose(C_atom @ N_gas, 0.0, atol=1e-14) and np.allclose(N_gas.T @ N_gas, np.eye(2), atol=1e-14)
assert N_aqu.shape == (6, 3) and np.allclose(C_aq @ N_aqu, 0.0, atol=1e-14) and np.allclose(N_aqu.T @ N_aqu, np.eye(3), atol=1e-14)


Exercise 3: Implement stoichiometric_nullspace_basis From Scratch
Methanol gas system nullspace shape: (5, 2) | ||C N||_max = 2.22e-16
Aqueous carbonate nullspace shape  : (6, 3) | ||C N||_max = 4.44e-16


**Discussion.** The SVD right singular vectors corresponding to zero singular values give an orthonormal basis of the space of reactions $\mathcal{N}(\mathbf{C})$ of any chemical or electrochemical system.